# 04 - RAG Evaluation: Retrieval & Factuality Metrics

This notebook evaluates the RAG system's retrieval quality and answer factuality.
We compute:
- Retrieval metrics: recall@k, MRR, nDCG (using drafted answers as proxy for relevance).
- Factuality metrics: faithfulness (fraction of supported claims), hallucination rate, attribution coverage.

Steps:
1. Load retrieved contexts and draft answers.
2. Decompose answers into atomic claims.
3. Verify each claim against retrieved chunks (NLI + QA).
4. Aggregate metrics per question and overall.

Outputs: `data/processed/ground_truth_eval.json`, metrics summary.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
import logging
from pathlib import Path
from typing import List, Dict, Any

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Paths
DATA_PROCESSED = Path('../data/processed')
RETRIEVED_CONTEXTS_PATH = DATA_PROCESSED / 'retrieved_contexts.json'
DRAFT1_PATH = DATA_PROCESSED / 'draft1_answers.json'
EVAL_OUTPUT_PATH = DATA_PROCESSED / 'ground_truth_eval.json'

# Imports
from biotech_rag.evaluation.claim_verification import (
    decompose_claims,
    verify_claim_hybrid,
    find_best_supporting_chunk
)
from biotech_rag.generation.llm_clients import get_openrouter_llm
try:
    from biotech_rag.config import settings
except ImportError:
    settings = None

In [ ]:
# Load data
with open(RETRIEVED_CONTEXTS_PATH, 'r', encoding='utf-8') as f:
    retrieved_records = json.load(f)

with open(DRAFT1_PATH, 'r', encoding='utf-8') as f:
    draft_records = json.load(f)

# Merge by row_id
retrieved_dict = {r['row_id']: r for r in retrieved_records}
merged_records = []
for d in draft_records:
    row_id = d['row_id']
    if row_id in retrieved_dict:
        merged = {**retrieved_dict[row_id], **d}
        merged_records.append(merged)

logger.info(f"Loaded {len(merged_records)} merged records for evaluation.")
print(f"Sample merged record keys: {list(merged_records[0].keys()) if merged_records else 'None'}")

In [ ]:
# Claim decomposition and verification
llm = get_openrouter_llm()
eval_results = []

for rec in merged_records[:5]:  # Sample for testing; remove [:5] for full run
    answer_text = rec.get('draft_answer', '')
    retrieved_chunks = rec.get('retrieved_chunks', [])
    
    if not answer_text or not retrieved_chunks:
        continue
    
    # Decompose
    claims = decompose_claims(answer_text, llm=llm)
    logger.info(f"Decomposed {len(claims)} claims for row {rec['row_id']}")
    
    claim_verifications = []
    max_claims = settings.max_claims_per_answer if settings else 5
    for claim in claims[:max_claims]:
        best_idx, result = find_best_supporting_chunk(claim, retrieved_chunks, method='hybrid')
        claim_verifications.append({
            'claim': claim,
            'supported': result.get('supported', False),
            'best_chunk_idx': best_idx,
            'verification_details': result
        })
    
    # Metrics
    supported_claims = sum(1 for c in claim_verifications if c['supported'])
    faithfulness = supported_claims / len(claims) if claims else 0.0
    hallucination_rate = 1.0 - faithfulness
    attribution_coverage = supported_claims / len(retrieved_chunks) if retrieved_chunks else 0.0  # Proxy
    
    eval_results.append({
        **rec,
        'claims': claims,
        'claim_verifications': claim_verifications,
        'faithfulness': faithfulness,
        'hallucination_rate': hallucination_rate,
        'attribution_coverage': attribution_coverage
    })

# Save
with open(EVAL_OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(eval_results, f, ensure_ascii=False, indent=2)

logger.info(f"Saved evaluation results to {EVAL_OUTPUT_PATH}")
print(f"Evaluated {len(eval_results)} records.")

In [ ]:
# Aggregate metrics summary
if eval_results:
    df_eval = pd.DataFrame(eval_results)
    summary = {
        'total_records': len(df_eval),
        'avg_faithfulness': df_eval['faithfulness'].mean(),
        'avg_hallucination_rate': df_eval['hallucination_rate'].mean(),
        'avg_attribution_coverage': df_eval['attribution_coverage'].mean(),
        'claims_per_answer': df_eval['claims'].apply(len).mean()
    }
    print("Evaluation Summary:")
    for k, v in summary.items():
        print(f"  {k}: {v:.3f}")
else:
    print("No evaluation results to summarize.")